# 04 — Playwright Code Generation

This notebook loads the mapped IR produced by Notebook 03 and generates a
Playwright Python test file.

Pipeline:
NL → IR → Mapped IR → Code → Execution → Analysis


In [15]:
from pathlib import Path
import json


## 1. Load Mapped IR


In [16]:
ir_path = Path("../sample-data/ir_examples/todomvc_mapped_ir.json")

with open(ir_path, "r") as f:
    mapped_ir = json.load(f)

mapped_ir


{'test_name': 'todomvc_add_and_complete_items',
 'description': 'Add “buy milk”, add “get gas in Susie’s car”, then complete “buy milk”, and complete “get gas in Susie’s car”.\n',
 'steps': [{'action': 'input',
   'target': 'new_todo_input',
   'value': 'buy milk',
   'selector': 'input.new-todo'},
  {'action': 'press',
   'target': 'new_todo_input',
   'key': 'Enter',
   'selector': 'input.new-todo'},
  {'action': 'input',
   'target': 'new_todo_input',
   'value': 'get gas in Susie’s car',
   'selector': 'input.new-todo'},
  {'action': 'press',
   'target': 'new_todo_input',
   'key': 'Enter',
   'selector': 'input.new-todo'},
  {'action': 'click', 'target': 'todo_item_1_checkbox', 'selector': None},
  {'action': 'click', 'target': 'todo_item_2_checkbox', 'selector': None}],
 'metadata': {'priority': 'high'}}

## 2. Playwright Code Templates

We define small helper functions to convert IR steps into Playwright code.


In [17]:
def generate_enter_text(step):
    return f'    await page.fill("{step["selector"]}", "{step["value"]}")'

def generate_press_enter(step):
    return f'    await page.press("{step["selector"]}", "Enter")'

def generate_assert_visible(step):
    return f'    await expect(page.locator("{step["selector"]}")).to_be_visible()'


## 3. Step Dispatcher

This maps IR actions → Playwright code generator functions.


In [18]:
ACTION_MAP = {
    "enter_text": generate_enter_text,
    "press_enter": generate_press_enter,
    "assert_visible": generate_assert_visible,
}


## 4. Generate Playwright Test Code


In [19]:
def generate_test_code(ir):
    test_name = ir["test_name"]

    # Header: imports + async test + async with block
    header = [
        "from playwright.async_api import async_playwright, expect",
        "import asyncio",
        "",
        f"async def test_{test_name}():",
        "    async with async_playwright() as p:",
        "        browser = await p.chromium.launch(headless=False)",
        "        page = await browser.new_page()",
        '        await page.goto("https://demo.playwright.dev/todomvc")',
        ""
    ]

    # Body: each step must be indented 8 spaces
    body = []
    for step in ir["steps"]:
        action = step["action"]
        selector = step["selector"]

        if action == "enter_text":
            body.append(f'        await page.fill("{selector}", "{step["value"]}")')

        elif action == "press_enter":
            body.append(f'        await page.press("{selector}", "Enter")')

        elif action == "assert_visible":
            body.append(f'        await expect(page.locator("{selector}")).to_be_visible()')

        else:
            body.append(f"        # Unsupported action: {action}")

    # Footer: close browser + main entrypoint
    footer = [
        "",
        "        await browser.close()",
        "",
        "if __name__ == '__main__':",
        f"    asyncio.run(test_{test_name}())"
    ]

    # Join all parts into final code
    return "\n".join(header + body + footer)


## 5. Render the Test Code


In [20]:
test_code = generate_test_code(mapped_ir)
print(test_code)


from playwright.async_api import async_playwright, expect
import asyncio

async def test_todomvc_add_and_complete_items():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto("https://demo.playwright.dev/todomvc")

        # Unsupported action: input
        # Unsupported action: press
        # Unsupported action: input
        # Unsupported action: press
        # Unsupported action: click
        # Unsupported action: click

        await browser.close()

if __name__ == '__main__':
    asyncio.run(test_todomvc_add_and_complete_items())


## 6. Save Test File


In [21]:
output_dir = Path("../generated-tests")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{mapped_ir['test_name']}_test.py"

with open(output_path, "w") as f:
    f.write(test_code)

output_path


PosixPath('../generated-tests/todomvc_add_and_complete_items_test.py')